# Exact (Hill) satellite on the grid of the satellite-start map (22/10/2026)

Report v2, big error 1, "TO DO for v3": seed `lyapunov_heatmap_satellite_start.ipynb` with the exact harmonic-balance satellite instead of the analytic one (`satellite_hb`).
The analytic seed exists only above an analytic boundary that is 8–15% below the true fold, and between the fold and about 1.2 × the fold it is too far from the satellite (1.1–1.4 in $x^\lambda$ units at 40–50 MHz, $\varepsilon_d = 10.12\,g_2$) and ends on the cat.

**What this notebook does.** For every $\varepsilon_d/g_2$ of the map ($2{:}0.1{:}13.5$) the period-1 satellite is followed by pseudo-arclength from the top of the map ($\varepsilon_p = 1.1$, $g_2/2\pi = 122$ MHz) towards lower $g_2$, down to its **fold** (saddle-node of limit cycles, where it is born together with the saddle, `Hill_method_satellite_fold.ipynb`).
The state at $\tau = 0$ is stored at every $\varepsilon_p$ of the map ($0.2{:}0.01{:}1.1$) above the fold; below the fold the satellite does not exist and the entry is `null`.
The fold sets the lower threshold in $g_2$: steps of the arclength that pass it (the parameter $s$ turns back) are rejected and halved, so the branch is never continued onto the saddle.

**Start.** The 60 MHz satellite at $\varepsilon_d = 10.12\,g_2$ (converged from the analytic seed, as in `Hill_method_satellite_fold`), continued in $g_2$ up to 122 MHz, then in $\varepsilon_d/g_2$ along the top of the map.
It is the member $\sigma = +1$ of the pair $a \to -a$, the same member as the old analytic seed.

**Conventions** (as in the other Hill notebooks): $g_2 = E_J\sin\varepsilon_p\,\varphi_a^2\varphi_b$, $\varepsilon_d = (\varepsilon_d/g_2)\,g_2$, $\kappa_b = \pi/10.4$ (code), $\tau = \omega_a t$, $x^\lambda = \varphi x$, $N_H = 16$.

**Output** `data/hill_orbits/satellite_seed_hill_22_10_2026.json`: `state[j][i]` $= [x_a, x_b, dx_a/d\tau, dx_b/d\tau]$ at $\tau = 0$ for ratio index `j` and $\varepsilon_p$ index `i`, the four Floquet multipliers (`max_abs_mu`, leading multiplier), and the fold `g2_fold_MHz[j]`.
In Julia (`SystemATS`, $t = 0$): $a = (x_a + i\,dx_a/d\tau)/(\sqrt2\varphi_a)$, $b = (x_b + i\,(dx_b/d\tau)/2)/(\sqrt2\varphi_b)$, then `system_from_amplitudes(a, b)`.
The whole notebook runs headless in about 5 min.

In [ ]:
import json, time, warnings
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import root
from scipy.integrate import solve_ivp
import sys
sys.path.insert(0, "/Users/tmalas/Desktop/Dev/Simulations_local/Code_python")
import Plot_setting
from Plot_setting import *

OUT_DIR = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/data/"
FIG_DIR = "/Users/tmalas/Desktop/Dev/Simulations_local/classical_simulations/figures/hill_orbits/"

w_a = 25.338776456203686
w_b = 2*w_a
phi_a, phi_b = 0.11, 0.204
E_J = 37.12*2*np.pi*1.5
kappa_b = np.pi/10.4                 # 1/ns, code convention (phi_b'' = ... - 2 kappa_b phi_b')
G0 = E_J*phi_a**2*phi_b              # g2 = s G0, s = sin(eps_p)
n = 2
M = np.diag([1.0, 0.25])
K = np.eye(2)

def MHz(g):
    return g/2/np.pi*1e3

## Harmonic balance (copied from `Hill_method_satellite_fold.ipynb`)
Exact one-period AFT. `strobe(z, N_H)` is the state $[x_a, x_b, dx_a/d\tau, dx_b/d\tau]$ at $\tau = 0$, `mu_monodromy` the four Floquet multipliers, `full_rhs` the equations of motion in $\tau$ (for the return check).

In [ ]:
class HB:
    def __init__(self, N_H, kfac=1.0, N=None):
        self.N_H, self.N = N_H, N or 64*N_H
        self.C = np.diag([0.0, kfac*kappa_b/w_b])
        dim = n*(2*N_H + 1)
        self.L = np.zeros((dim, dim))
        self.L[:n, :n] = K
        for k in range(1, N_H + 1):
            A, Bk = K - k**2*M, k*self.C
            s_, c_ = 2*k - 1, 2*k
            self.L[s_*n:(s_+1)*n, s_*n:(s_+1)*n] = A
            self.L[s_*n:(s_+1)*n, c_*n:(c_+1)*n] = -Bk
            self.L[c_*n:(c_+1)*n, s_*n:(s_+1)*n] = Bk
            self.L[c_*n:(c_+1)*n, c_*n:(c_+1)*n] = A
        t = np.linspace(0, 2*np.pi, self.N, endpoint=False)
        k = np.arange(1, N_H + 1)
        Phi = np.empty((self.N, 2*N_H + 1))
        Phi[:, 0] = 1/np.sqrt(2)
        Phi[:, 1::2] = np.sin(np.outer(t, k))
        Phi[:, 2::2] = np.cos(np.outer(t, k))
        self.Gamma = np.kron(Phi, np.eye(n))
        self.Gamma_pinv = (2/self.N)*self.Gamma.T

    def set_s(self, s, r):
        self.B = 2*E_J*s*np.array([phi_a**2/w_a, phi_b**2/w_b])          # (beta_a', beta_b')
        self.b_ext = np.zeros(n*(2*self.N_H + 1))
        self.b_ext[2*2*n + 1] = -np.sqrt(2)*r*s*G0*phi_b/w_b             # c_2 block of mode b

    def X(self, z):
        return (self.Gamma @ z).reshape(self.N, n).T

    def residual(self, z):
        X = self.X(z)
        Fnl = -np.sqrt(2)*self.B[:, None]*(np.cos(np.sqrt(2)*(X[0] + X[1])) - 1)[None, :]
        return self.L @ z + self.Gamma_pinv @ Fnl.T.reshape(-1) - self.b_ext

    def jacobian(self, z):
        X = self.X(z)
        sn = np.sin(np.sqrt(2)*(X[0] + X[1]))
        JG = np.einsum('j,ab,jbm->jam', 2*sn, np.outer(self.B, [1, 1]),
                       self.Gamma.reshape(self.N, n, -1)).reshape(n*self.N, -1)
        return self.L + self.Gamma_pinv @ JG

    def solve(self, z0, tol=1e-12):
        sol = root(self.residual, z0, jac=self.jacobian, method="hybr", tol=tol, options={"maxfev": 100*(len(z0) + 1)})
        if not sol.success:
            raise RuntimeError(sol.message)
        return sol.x


def orbit(z, N_H, t):
    k = np.arange(1, N_H + 1)
    out = []
    for zz in (z[::2], z[1::2]):
        s_, c_ = zz[1::2], zz[2::2]
        x = zz[0]/np.sqrt(2) + np.sin(np.outer(t, k)) @ s_ + np.cos(np.outer(t, k)) @ c_
        v = np.cos(np.outer(t, k)) @ (k*s_) - np.sin(np.outer(t, k)) @ (k*c_)
        out.append((x, v))
    return out

def strobe(z, N_H, tau=0.0):
    (x_a, v_a), (x_b, v_b) = orbit(z, N_H, np.array([tau]))
    return np.array([x_a[0], x_b[0], v_a[0], v_b[0]])

def memory_phase(z):
    za = z[::2]
    return np.angle(za[2] + 1j*za[1])/np.pi

def mu_monodromy(hbk, z):
    # the four Floquet multipliers of the HB orbit z (variational equation over one period)
    Minv, base = np.linalg.inv(M), np.outer(hbk.B, [1, 1])
    def rhs(t, Y):
        (x_a, _), (x_b, _) = orbit(z, hbk.N_H, np.atleast_1d(t))
        A = np.zeros((4, 4)); A[:2, 2:] = np.eye(2)
        A[2:, :2] = -Minv @ (K + 2*np.sin(np.sqrt(2)*(x_a[0] + x_b[0]))*base); A[2:, 2:] = -Minv @ hbk.C
        return (A @ Y.reshape(4, 4)).reshape(-1)
    sol = solve_ivp(rhs, (0, 2*np.pi), np.eye(4).reshape(-1), method="DOP853", rtol=1e-10, atol=1e-12)
    return np.linalg.eigvals(sol.y[:, -1].reshape(4, 4))

def full_rhs(s, r, kfac=1.0):
    beta = 2*E_J*s*np.array([phi_a**2/w_a, phi_b**2/w_b])
    eps_d_p, kp = r*s*G0*phi_b/w_b, kfac*kappa_b/w_b
    def f(t, Y):
        x_a, x_b, v_a, v_b = Y
        g = np.cos(np.sqrt(2)*(x_a + x_b)) - 1
        return [v_a, v_b, -x_a + np.sqrt(2)*beta[0]*g, 4*(-x_b - np.sqrt(2)*eps_d_p*np.cos(2*t) + np.sqrt(2)*beta[1]*g - kp*v_b)]
    return f

## Analytic satellite (only used once, to converge the 60 MHz satellite at $\varepsilon_d = 10.12\,g_2$)

In [ ]:
yb_grid = np.linspace(1e-3, 3.0, 3000)

def satellite_hb(s, r, kfac=1.0):
    kp = kfac*kappa_b/w_b
    ba, bb = 2*E_J*s*phi_a**2/w_a, 2*E_J*s*phi_b**2/w_b
    ed, bS = r*s*G0*phi_b/w_b, ba + bb
    gam = 1 + bb/(ba*(1 + kp**2))
    eeff = lambda y: np.sqrt(np.maximum(ed**2 - 4*kp**2*y**2, 0.0))
    def y_of(yb):
        y = yb + ba*ed/(2*bb)
        for _ in range(30):
            y = yb + ba*eeff(y)/(2*bb)
        return y
    def u_of(yb):
        eta = 2*bb*yb
        return (np.sqrt((1 + eta)**2 + 4*eta**2) - (1 + eta))/(2*eta)
    def x2_of(yb):
        u = u_of(yb)
        return (2*(1 + u)*yb**2 - eeff(y_of(yb))/bb)/(1 + 2*u)
    def resid(yb):
        u = u_of(yb)
        return yb*(1 + u) - 2*bS*((1 + (1 + u)**2/2)*yb**2 + (1 + u**2)*x2_of(yb))
    rv = resid(yb_grid)
    for k in np.where(rv[:-1]*rv[1:] <= 0)[0]:
        lo, hi = yb_grid[k], yb_grid[k + 1]
        for _ in range(60):
            mid = (lo + hi)/2
            if resid(lo)*resid(mid) <= 0:
                hi = mid
            else:
                lo = mid
        yb = (lo + hi)/2
        x2 = x2_of(yb)
        if x2 <= 0:
            continue
        u, y, x = u_of(yb), y_of(yb), np.sqrt(x2)
        if 2*kp*abs(y) > ed:
            continue
        th = -np.arcsin(2*kp*y/ed)/2
        a1 = -(1 + u)*yb/(2*gam)
        return dict(a0=x*np.exp(1j*th), a1=a1, am1=-ba*eeff(y)/(2*bb)*np.exp(2j*th),
                    b0=y*np.exp(2j*th), bm1=u*x*np.exp(3j*th), b2=bb/ba*a1)
    return None

def seed_z(S, N_H):
    za, zb = np.zeros(2*N_H + 1), np.zeros(2*N_H + 1)
    za[0] = 2*np.real(S["a1"])
    za[1], za[2] = np.sqrt(2)*np.imag(S["a0"]), np.sqrt(2)*np.real(S["a0"])
    za[3], za[4] = np.sqrt(2)*np.imag(S["am1"]), np.sqrt(2)*np.real(S["am1"])
    zb[0] = 2*np.real(S["b2"])
    zb[3], zb[4] = np.sqrt(2)*np.imag(S["b0"]), np.sqrt(2)*np.real(S["b0"])
    zb[5], zb[6] = np.sqrt(2)*np.imag(S["bm1"]), np.sqrt(2)*np.real(S["bm1"])
    z = np.zeros(n*(2*N_H + 1)); z[::2], z[1::2] = za, zb
    return z


def solve_robust(hbk, z):
    # Powell sometimes stalls at tol = 1e-12 when the guess is already converged to ~1e-8
    try:
        return hbk.solve(z)
    except RuntimeError:
        return hbk.solve(z, tol=1e-9)

## Grid of the map, and the satellite along its top edge
Same grid as `lyapunov_heatmap_satellite_start.ipynb`: `vcat(0.2:.01:1.1)` and `2:.1:13.5`.

In [ ]:
eps_p_list = np.round(np.arange(0.2, 1.1 + 1e-9, 0.01), 2)
ratio_list = np.round(np.arange(2.0, 13.5 + 1e-9, 0.1), 1)
s_grid = np.sin(eps_p_list)
s_top = s_grid[-1]

N_H = 16
r0, g_seed = 10.12, 60.0
hb = HB(N_H)
s_seed = g_seed/MHz(G0)
hb.set_s(s_seed, r0)
z_sat60 = hb.solve(seed_z(satellite_hb(s_seed, r0), N_H))

z = z_sat60.copy()                                      # up to the top of the map at 10.12
for s_new in np.linspace(s_seed, s_top, 25)[1:]:
    hb.set_s(s_new, r0); z = hb.solve(z)
z_top = {}                                              # then along the top edge, in eps_d/g2
for part in (ratio_list[ratio_list > r0], ratio_list[ratio_list < r0][::-1]):
    zz, r_prev = z.copy(), r0
    for r in part:
        for rr in np.linspace(r_prev, r, 4)[1:]:
            hb.set_s(s_top, rr); zz = hb.solve(zz)
        z_top[r] = zz.copy(); r_prev = r
ph = np.array([memory_phase(z_top[r]) for r in ratio_list])
print(f"satellite at g2/2pi = {MHz(s_top*G0):.1f} MHz for all {len(ratio_list)} ratios; memory phase {ph.min():+.3f} .. {ph.max():+.3f} pi")

## Continuation down to the fold, at every $\varepsilon_d/g_2$
`satellite_column` runs the arclength in $(z, s)$ towards lower $s$. Each grid value of $s$ crossed by a step is solved at fixed $s$ from the linear interpolation between the two ends of the step (and must stay within 1% of it).
A step after which $s$ increases has passed the fold: it is rejected and $ds$ halved, until $ds < 10^{-6}$. The last accepted point gives the fold.
The Floquet multipliers are then computed at every grid point (monodromy matrix).

In [ ]:
def satellite_column(hb, z, s, r, s_grid, ds_max=0.01, ds_fold=1e-6):
    """Pseudo-arclength along the satellite branch at fixed eps_d/g2 = r, from (z, s) towards lower s, down to the fold.
    Returns the HB coefficients at every s of s_grid (NaN rows below the fold or above s), the s of the fold, the number of steps.
    A step that passes the fold (s turns back) is rejected and the step halved, so the last accepted point is within ds_fold of the fold."""
    def F(y):
        hb.set_s(y[-1], r); return hb.residual(y[:-1])
    def dF(y):
        hb.set_s(y[-1], r); Jz, r_ = hb.jacobian(y[:-1]), hb.residual(y[:-1])
        h = 1e-7
        hb.set_s(y[-1] + h, r)
        return np.column_stack([Jz, (hb.residual(y[:-1]) - r_)/h])
    def tangent(y, t_prev=None):
        t = np.linalg.svd(dF(y))[2][-1]
        if (t_prev is not None and t @ t_prev < 0) or (t_prev is None and t[-1] > 0):
            t = -t
        return t
    zs = np.full((len(s_grid), len(z)), np.nan)
    on_grid = np.isclose(s_grid, s, rtol=0, atol=1e-12)
    zs[on_grid] = z
    y = np.append(z, s); t = tangent(y); ds = 0.003; steps = 0
    while True:
        sol = root(lambda w: np.append(F(w), t @ (w - y) - ds), y + ds*t,
                   jac=lambda w: np.vstack([dF(w), t]), method="hybr", tol=1e-12)
        t_new = tangent(sol.x, t) if sol.success else None
        if not sol.success or t_new[-1] > 0 or sol.x[-1] >= y[-1]:   # failed, or passed the fold
            ds /= 2
            if ds < ds_fold:
                break
            continue
        steps += 1
        for g in np.where((s_grid >= sol.x[-1]) & (s_grid < y[-1]))[0]:   # grid values of s crossed by this step
            w = (y[-1] - s_grid[g])/(y[-1] - sol.x[-1])
            guess = y[:-1] + w*(sol.x[:-1] - y[:-1])
            hb.set_s(s_grid[g], r); zs[g] = solve_robust(hb, guess)
            assert np.linalg.norm(zs[g] - guess) < 1e-2*np.linalg.norm(guess), "grid solve left the branch"
        y, t = sol.x, t_new; ds = min(1.3*ds, ds_max)
        if y[-1] < s_grid.min():
            return zs, np.nan, steps                 # no fold above the lowest grid value
    return zs, y[-1], steps

In [ ]:
n_r, n_p = len(ratio_list), len(eps_p_list)
state = np.full((n_r, n_p, 4), np.nan)        # [x_a, x_b, dx_a/dtau, dx_b/dtau] at tau = 0
mu_all = np.full((n_r, n_p, 4), np.nan + 0j)
s_fold = np.full(n_r, np.nan)
t0 = time.time()
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)   # overflows inside rejected Powell iterations
    for j, r in enumerate(ratio_list):
        zs, s_fold[j], _ = satellite_column(hb, z_top[r], s_top, r, s_grid)
        for i in np.where(np.isfinite(zs[:, 0]))[0]:
            state[j, i] = strobe(zs[i], N_H)
            hb.set_s(s_grid[i], r); mu_all[j, i] = mu_monodromy(hb, zs[i])
g2_fold = MHz(s_fold*G0)
max_abs_mu = np.abs(mu_all).max(axis=2)
mu_lead = np.take_along_axis(mu_all, np.nanargmax(np.abs(np.where(np.isnan(mu_all), 0, mu_all)), axis=2)[..., None], axis=2)[..., 0]
has = np.isfinite(state[..., 0])
print(f"{time.time() - t0:.0f} s; satellite on {has.sum()}/{has.size} grid points ({100*has.mean():.1f}%), "
      f"stable (max|mu| < 1) on {(max_abs_mu < 1).sum()}")
print(f"satellite fold: {g2_fold.max():.2f} MHz (eps_d/g2 = {ratio_list[0]}) .. {g2_fold.min():.2f} MHz (eps_d/g2 = {ratio_list[-1]})")

## Checks
1. The fold against `satellite_fold_line_20_10_2026.json` (`Hill_method_satellite_fold`, ratios $2{:}0.5{:}13.5$, on this grid).
2. The lowest grid point of each column is the satellite, not the saddle: no real multiplier above 1 there.
3. The stored state is on the orbit: time integration of the full equations over one period from it, at the lowest grid point of each column.
4. Stability along $\varepsilon_d = 10.1\,g_2$: the satellite should be stable from the fold (40.4 MHz) to its period doubling ($\approx 75$ MHz), and again above $\approx 104$ MHz.

In [ ]:
with open(OUT_DIR + "hill_orbits/satellite_fold_line_20_10_2026.json") as f:
    ref = json.load(f)
ref_r, ref_g = np.array(ref["eps_d_over_g2"]), np.array(ref["g2_fold_MHz_hill"])
mine = np.array([g2_fold[np.argmin(np.abs(ratio_list - r))] for r in ref_r])
print(f"1. fold vs satellite_fold_line_20_10_2026: max |difference| = {np.max(np.abs(mine - ref_g)):.3f} MHz")

i_low = np.argmax(has, axis=1)                         # lowest eps_p with a satellite, per ratio
mu_low = mu_all[np.arange(n_r), i_low]
real_above_1 = np.any((np.abs(mu_low.imag) < 1e-9) & (mu_low.real > 1), axis=1)
print(f"2. lowest grid point above the fold: max|mu| = {np.abs(mu_low).max(axis=1).min():.3f} .. {np.abs(mu_low).max(axis=1).max():.3f}, "
      f"real multiplier > 1 (saddle) at {real_above_1.sum()} ratios; gap to the fold {np.min(MHz(s_grid[i_low]*G0) - g2_fold):.2f} .. {np.max(MHz(s_grid[i_low]*G0) - g2_fold):.2f} MHz")

err = []
for j, r in enumerate(ratio_list):
    y0 = state[j, i_low[j]]
    y1 = solve_ivp(full_rhs(s_grid[i_low[j]], r), (0, 2*np.pi), y0, method="DOP853", rtol=1e-11, atol=1e-13).y[:, -1]
    err.append(np.linalg.norm(y1 - y0)/np.linalg.norm(y0))
print(f"3. one-period return error, relative: max {np.max(err):.1e}")

j = np.argmin(np.abs(ratio_list - 10.1))
st = max_abs_mu[j] < 1
edges = np.where(np.diff(st.astype(int)) != 0)[0]
print(f"4. eps_d/g2 = {ratio_list[j]}: stability changes between g2/2pi = " +
      ", ".join(f"{MHz(s_grid[e]*G0):.1f}-{MHz(s_grid[e + 1]*G0):.1f} MHz" for e in edges) + f" (fold at {g2_fold[j]:.2f} MHz)")

In [ ]:
g2_MHz_list = MHz(s_grid*G0)
alpha_sq = ratio_list/2
fig, axes = plt.subplots(1, 2, figsize=(2*figsize[0], figsize[1]))
ax = axes[0]
amp = np.where(has, np.hypot(state[..., 0], state[..., 2])/(np.sqrt(2)*phi_a), np.nan)   # |a(0)|
pc = ax.pcolormesh(g2_MHz_list, alpha_sq, amp, cmap="viridis", shading="nearest")
fig.colorbar(pc, ax=ax, label=r"$|a(0)|$")
ax.set_title("Seed: exact satellite at $t = 0$", fontsize=8)
ax = axes[1]
pc = ax.pcolormesh(g2_MHz_list, alpha_sq, np.log10(max_abs_mu), cmap="RdBu_r", vmin=-0.5, vmax=0.5, shading="nearest")
fig.colorbar(pc, ax=ax, label=r"$\log_{10}\max|\mu|$")
ax.contour(g2_MHz_list, alpha_sq, np.where(has, max_abs_mu, 0.5), levels=[1], colors="k", linewidths=0.6)
ax.set_title("Floquet stability of the satellite", fontsize=8)
for ax in axes:
    ax.plot(g2_fold, alpha_sq, "k--", lw=1, label="satellite fold (Hill)")
    ax.set_xlabel(r"$g_2/2\pi$ (MHz)"); ax.set_ylabel(r"$\varepsilon_d/2g_2$")
    ax.set_xlim(g2_MHz_list.min(), g2_MHz_list.max()); ax.set_ylim(alpha_sq.min(), alpha_sq.max())
axes[0].legend(fontsize=5, loc="upper right")
plt.tight_layout()
plt.savefig(FIG_DIR + "satellite_seed_hill_grid_22_10_2026.pdf")
plt.show()

## Export
`null` below the fold. Outer index: $\varepsilon_d/g_2$ (`j`), inner index: $\varepsilon_p$ (`i`), i.e. `state[j][i]`, as `Lyap_coef_chaos[j, i]` in the Julia map.

In [ ]:
nan2none = lambda a: [[None if not np.all(np.isfinite(v)) else (v.tolist() if np.ndim(v) else float(v)) for v in row] for row in a]
with open(OUT_DIR + "hill_orbits/satellite_seed_hill_22_10_2026.json", "w") as f:
    json.dump({"description": "Exact period-1 satellite (harmonic balance, pseudo-arclength from the top of the map down to its fold) "
                              "on the grid of lyapunov_heatmap_satellite_start.ipynb; Hill_satellite_seed_grid.ipynb",
               "g2_definition": "E_J sin(eps_p) phi_a^2 phi_b", "kappa_b": kappa_b, "N_H": N_H,
               "state_definition": "[x_a, x_b, dx_a/dtau, dx_b/dtau] at tau = omega_a t = 0, x^lambda = phi x; "
                                   "a = (x_a + i dx_a/dtau)/(sqrt2 phi_a), b = (x_b + i (dx_b/dtau)/2)/(sqrt2 phi_b); null below the fold",
               "eps_p_list": eps_p_list.tolist(), "eps_d_over_g2": ratio_list.tolist(), "g2_MHz": g2_MHz_list.tolist(),
               "state": nan2none(state), "max_abs_mu": nan2none(max_abs_mu),
               "mu_lead_re": nan2none(mu_lead.real), "mu_lead_im": nan2none(mu_lead.imag),
               "s_fold": s_fold.tolist(), "g2_fold_MHz": g2_fold.tolist()}, f)
print("written", OUT_DIR + "hill_orbits/satellite_seed_hill_22_10_2026.json")